In [1]:
# ===== 0. 설정 (커널 재시작하면 이것부터 실행) =====
import torch, cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

DEVICE   = 0 if torch.cuda.is_available() else "cpu"
SEG_ROOT = Path("archive/PKLot/PKLotSegmented")
DATA_DIR = Path("datasets/pklot_cls")
BEST_PT  = "runs/classify/runs/cls_v1/weights/best.pt"

best = YOLO(BEST_PT)
print("device:", DEVICE, "| classes:", best.names)

C:\Users\HP\anaconda3\envs\SmartPkLot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: 0 | classes: {0: 'empty', 1: 'occupied'}


In [6]:
import numpy as np
rows = [{"file": str(f), "label": f.parent.name}
        for f in (DATA_DIR / "test").rglob("*.jpg")]
test_df = pd.DataFrame(rows)

# test_df는 PUC 전체 (file, label, weather, pred ...)
test_df["date"] = test_df["file"].map(lambda f: Path(f).name.removeprefix("PUC_")[:10])
dates = sorted(test_df["date"].unique())
print("PUC 날짜 수:", len(dates))

rng = np.random.default_rng(0)
shuffled = [str(d) for d in rng.permutation(dates)]   # 날짜 순서를 무작위로 섞기

K = 5                                     # 현장 데이터로 넣을 날 수 (1, 3, 5로 바꿔가며)
ft_dates   = set(shuffled[:K])            # train에 넣을 날
val_dates  = set(shuffled[5:6])           # val에 넣을 날 (K와 상관없이 고정)
test_dates = set(shuffled[6:])            # test (K와 상관없이 고정)

print("train 추가:", sorted(ft_dates))
print("val 추가  :", sorted(val_dates))
print("test 날짜 수:", len(test_dates))

PUC 날짜 수: 38
train 추가: ['2012-09-13', '2012-09-15', '2012-09-21', '2012-10-15', '2012-10-16']
val 추가  : ['2012-09-14']
test 날짜 수: 32


In [7]:
display(test_df.head())

,file,label,date
0,datasets\pklot_cls\test\empty\PUC_2012-09-11_1...,empty,2012-09-11
1,datasets\pklot_cls\test\empty\PUC_2012-09-11_1...,empty,2012-09-11
2,datasets\pklot_cls\test\empty\PUC_2012-09-11_1...,empty,2012-09-11
3,datasets\pklot_cls\test\empty\PUC_2012-09-11_1...,empty,2012-09-11
4,datasets\pklot_cls\test\empty\PUC_2012-09-11_1...,empty,2012-09-11


In [9]:
import os, shutil

#DATA_DIR 첫 학습 데이더 셋 PATH
NEW = Path(f"datasets/pklot_cls_ft_puc{K}d")        # 실험 B 폴더

def link(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        try:
            os.link(src, dst)            # 하드링크: 복사 없이 같은 파일을 가리킴 (용량 0)
        except OSError:
            shutil.copy2(src, dst)       # 안 되면 복사

# (1) UFPR train, val은 그대로 가져오기
for split in ["train", "val"]:
    for f in tqdm(list((DATA_DIR / split).rglob("*.jpg")), desc=split):
        link(f, NEW / split / f.parent.name / f.name)

# (2) PUC는 날짜에 따라 나눠 넣기
for r in tqdm(test_df.itertuples(), total=len(test_df), desc="PUC"):
    if r.date in ft_dates:    split = "train"
    elif r.date in val_dates: split = "val"
    else:                     split = "test"
    link(Path(r.file), NEW / split / r.label / Path(r.file).name)

for split in ["train", "val", "test"]:
    for cls in ["empty", "occupied"]:
        print(split, cls, len(list((NEW / split / cls).glob("*.jpg"))))

PUC: 100%|██████████| 424223/424223 [17:16<00:00, 409.41it/s] 


train empty 154374
train occupied 143577
val empty 17760
val occupied 37076
test empty 185937
test occupied 157127


In [10]:
model_B = YOLO("yolo26n-cls.pt")
model_B.train(data=str(NEW), epochs=10, imgsz=96, batch=128, workers=2,
              cache="ram", device=DEVICE, project="runs", name=f"cls_ft_puc{K}d")
print("B best:", model_B.trainer.best)

New https://pypi.org/project/ultralytics/8.4.172 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.171  Python-3.11.16 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX A6000, 49140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets\pklot_cls_ft_puc5d, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=96, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, m

In [11]:
ft = test_df[test_df["date"].isin(ft_dates)].copy()
ft["frame"] = ft["file"].map(lambda f: Path(f).name.removeprefix("PUC_").split("#")[0])

print("추가한 날 수     :", len(ft_dates))
print("추가한 전체 사진 수:", ft["frame"].nunique())   # 네가 라벨링할 사진 수에 해당
print("추가한 크롭 수    :", len(ft))                  # 칸 단위 라벨 수
print(ft["label"].value_counts())

추가한 날 수     : 5
추가한 전체 사진 수: 701
추가한 크롭 수    : 70083
label
empty       40830
occupied    29253
Name: count, dtype: int64


In [12]:
model_A = YOLO("runs/classify/runs/cls_v1/weights/best.pt")   # 지금 모델 (UFPR만)
model_B = YOLO(model_B.trainer.best)

for name, m in [("A: UFPR만", model_A), (f"B: UFPR + PUC {K}일", model_B)]:
    r = m.val(data=str(NEW), split="test", imgsz=96, batch=1024,
              workers=8, device=DEVICE, verbose=False)
    print(f"{name:20s} test top1 = {r.top1:.5f}")

Ultralytics 8.4.171  Python-3.11.16 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX A6000, 49140MiB)
YOLO26n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs
train: D:\SmartParkingSystem\datasets\pklot_cls_ft_puc5d\train... found 297951 images in 2 classes  
val: D:\SmartParkingSystem\datasets\pklot_cls_ft_puc5d\val... found 54836 images in 2 classes  
test: D:\SmartParkingSystem\datasets\pklot_cls_ft_puc5d\test... found 343064 images in 2 classes  
test: Fast image access  (ping: 0.10.0 ms, read: 0.20.1 MB/s, size: 1.1 KB)
test: Scanning D:\SmartParkingSystem\datasets\pklot_cls_ft_puc5d\test... 343064 images, 0 corrupt: 100% ━━━━━━━━━━━━ 343064/343064 558.6it/s 10:140.1ss
test: New cache created: D:\SmartParkingSystem\datasets\pklot_cls_ft_puc5d\test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 336/336 2.8it/s 2:02<0.1ss9
                   all      0.998          1
Speed: 0.0ms preprocess, 0.0ms inference, 0.0ms loss, 0.0ms postprocess per 

#### 학습 비교 A vs B
 
- A의 경우에는 UFPR만 학습을 했고 정확도는 99.779% 오류율은 0.221% 오류 장수는 약 758장
- B의 경우에는 UFPR + PUC(700장) 정확도는 99.920% 오류율은 0.080% 오류 장수는 약 274장
- 둘의 차이는 약 500장 64%의 효율 개선.